In [2]:
%pip install -U langchain langchain-community langchain-core langchain-classic
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import SQLDatabase
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

C:\Users\Jafar tamboli\AppData\Local\Temp\ipykernel_7032\1751766401.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [10]:
import sys
print(sys.executable)
!{sys.executable} -m pip show langchain-core

d:\rag trial\.venv\Scripts\python.exe


'd:\rag' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
import pkgutil
import langchain_core

print([
    m.name
    for m in pkgutil.iter_modules(langchain_core.__path__)
])

['_api', '_import_utils', '_security', 'agents', 'caches', 'callbacks', 'chat_history', 'chat_loaders', 'chat_sessions', 'cross_encoders', 'document_loaders', 'documents', 'embeddings', 'env', 'example_selectors', 'exceptions', 'globals', 'indexing', 'language_models', 'load', 'messages', 'output_parsers', 'outputs', 'prompt_values', 'prompts', 'rate_limiters', 'retrievers', 'runnables', 'stores', 'structured_query', 'sys_info', 'tools', 'tracers', 'utils', 'vectorstores', 'version']


In [12]:
from langchain_core.output_parsers import StrOutputParser

print("Working!")

Working!


In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = Path("terms_and_conditions")

documents = []

for pdf_file in pdf_folder.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [5]:
from pathlib import Path
from pypdf import PdfReader

PDF_FOLDER = Path("documents")
pdf_files = list(PDF_FOLDER.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in: {PDF_FOLDER.resolve()}"
    )

pdf_path = pdf_files[0]
reader = PdfReader(str(pdf_path))

print("PDF:", pdf_path.name)
print("Total pages:", len(reader.pages))

PDF: Terms_and_Conditions.pdf
Total pages: 10


In [6]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    pages.append({
        "page_number": page_number,
        "text": text
    })

print("Pages extracted:", len(pages))

Pages extracted: 10


In [7]:
if pages:
    print(pages[0]["text"][:1500])
else:
    print("No pages were extracted.")

 
Draft for review • Complete placeholders before publication Page 1 
CAMPUS EATS 
CUSTOMER TERMS & CONDITIONS 
Campus-only food ordering for delivery, dine-in and pickup 
 
For students, faculty, staff and visitors at MIT ADT University 
Participating cafés, messes, food outlets and restaurants located within the campus only. 
Platform operator Jafar Tamboli 
Support contact 9699357027 
Website / app https://campus-eats-2znk.onrender.com/index.html 
Important: This is a practical draft for review, not legal advice. Before publishing, the operator should 
complete the placeholders and have the document reviewed for the actual business model, payment 
flow, campus permissions and applicable Indian law. These terms do not replace any mandatory 
consumer rights. 
Version 1.0 | Customer-facing policy draft 
At a glance 
These terms explain how Campus Eats works, what customers can expect, and what customers need to do when 
placing campus food orders. 
• Campus Eats is intended for orders 

In [8]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [9]:
full_text = "\n".join(
    page["text"] for page in pages
)

print("Total characters:", len(full_text))

Total characters: 30160


In [10]:
chunks = split_text(
    full_text,
    chunk_size=500,
    overlap=100
)

print("Total chunks:", len(chunks))

Total chunks: 76


In [11]:
print(chunks[0])

 
Draft for review • Complete placeholders before publication Page 1 
CAMPUS EATS 
CUSTOMER TERMS & CONDITIONS 
Campus-only food ordering for delivery, dine-in and pickup 
 
For students, faculty, staff and visitors at MIT ADT University 
Participating cafés, messes, food outlets and restaurants located within the campus only. 
Platform operator Jafar Tamboli 
Support contact 9699357027 
Website / app https://campus-eats-2znk.onrender.com/index.html 
Important: This is a practical draft for revi


In [12]:
print("End of first chunk:")
print(repr(chunks[0][-100:]))

print("\nBeginning of second chunk:")
print(repr(chunks[1][:100]))

End of first chunk:
' app https://campus-eats-2znk.onrender.com/index.html \nImportant: This is a practical draft for revi'

Beginning of second chunk:
' app https://campus-eats-2znk.onrender.com/index.html \nImportant: This is a practical draft for revi'
